# Yemen MarketShield AI — Walk-Forward Validation (Research Only)

**Purpose:** strengthen the evaluation of the experimental next-month wheat-price-increase flag using monthly walk-forward replay. This notebook does not establish operational readiness.

## Important limitations

- The target is a **greater-than-5% rise in the next month's `c_wheat` close estimate**, not a measured food crisis or a validated humanitarian outcome.
- The World Bank Real-Time Prices series mixes direct price data and model-estimated missing values, and its historical estimates can be updated. This notebook uses the frozen release dated `2026-03-24`; historical revisions create a possible vintage/point-in-time limitation.
- The target period 2022 onward was already viewed in earlier experiments. Therefore, these results are a corrected **retrospective walk-forward replay**, not a completely untouched independent test. A genuine final test requires independent observed prices or a future dataset vintage that was not used during model development.
- The included prediction CSVs are cached per-origin results for immediate reproduction of the tables. Set `RETRAIN_MODELS = True` to rerun the LightGBM fits from the source archives.
- Do not use this model for real-world humanitarian, purchasing, financial, or policy decisions. Any future use requires an independent evaluation, data freshness controls, probability/threshold calibration, and human review.

Sources: [World Bank food prices](https://microdata.worldbank.org/catalog/4508), [exchange rates](https://microdata.worldbank.org/catalog/6159), [energy prices](https://microdata.worldbank.org/catalog/6133).

## 1. Install / import dependencies

The notebook installs LightGBM only if it is unavailable.

In [1]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("lightgbm") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "lightgbm"])

import io
import zipfile
import os
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    balanced_accuracy_score, confusion_matrix
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)


## 2. Load the same three World Bank ZIP files

In Google Colab, upload the three source archives listed below. For a quick reproduction of the included metric tables, upload the two prediction-cache CSVs from this validation package in the same dialog. If the caches are absent, the notebook fits the two LightGBM variants at every forecast origin and may take several minutes. In local Jupyter, put all available files beside the notebook.

The stored per-origin prediction caches are derived outputs from earlier runs of the same prequential model configurations; their market/month keys and target labels are checked before comparison. They are not independent ground truth.

In [2]:
EXPECTED = {
    "food": "YEM_RTFP_mkt_2009_2026-03-24.csv",
    "fx": "YEM_RTFX_mkt_2009_2026-03-24.csv",
    "energy": "YEM_RTEP_mkt_2009_2026-03-24.csv",
}
EXPECTED_ZIPS = [
    "YEM_RTFP_mkt_2009_2026-03-24.zip",
    "YEM_RTFX_mkt_2009_2026-03-24.zip",
    "YEM_RTEP_mkt_2009_2026-03-24.zip",
]

try:
    from google.colab import files
    print("Upload the three World Bank ZIP archives listed in the project README.")
    uploaded = files.upload()
    # Save selected archives and optional cached prediction CSVs in the working directory.
    for uploaded_name, uploaded_contents in uploaded.items():
        Path(uploaded_name).write_bytes(uploaded_contents)
except ImportError:
    from pathlib import Path
    missing = [name for name in EXPECTED_ZIPS if not Path(name).is_file()]
    if missing:
        raise FileNotFoundError(
            "Missing source ZIP files: " + ", ".join(missing) +
            ". Download the version 2026-03-24 from the official source links."
        )
    uploaded = {name: Path(name).read_bytes() for name in EXPECTED_ZIPS}

def read_csv_from_uploaded_zip(expected_csv):
    matches = []
    for filename, contents in uploaded.items():
        if filename.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(contents)) as zf:
                candidates = [n for n in zf.namelist()
                              if os.path.basename(n) == expected_csv]
                for name in candidates:
                    with zf.open(name) as f:
                        matches.append(pd.read_csv(f, low_memory=False))
    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one ZIP containing {expected_csv}; found {len(matches)}"
        )
    return matches[0]

food = read_csv_from_uploaded_zip(EXPECTED["food"])
fx = read_csv_from_uploaded_zip(EXPECTED["fx"])
energy = read_csv_from_uploaded_zip(EXPECTED["energy"])
for frame in (food, fx, energy):
    frame["price_date"] = pd.to_datetime(frame["price_date"])

print("Food:", food.shape, "| FX:", fx.shape, "| Energy:", energy.shape)


Food: (5175, 165) | FX: (5175, 40) | Energy: (5175, 34)


## 3. Validate keys and merge model inputs

`Market Average` is excluded from modelling so the aggregate is not counted as a separate local market.

In [3]:
KEY = ["geo_id", "price_date"]
for name, frame in [("food", food), ("fx", fx), ("energy", energy)]:
    if frame.duplicated(KEY).any():
        raise ValueError(f"Duplicate market-month keys detected in {name}.")

food_keys = set(map(tuple, food[KEY].astype(str).to_numpy()))
fx_keys = set(map(tuple, fx[KEY].astype(str).to_numpy()))
energy_keys = set(map(tuple, energy[KEY].astype(str).to_numpy()))
print("Food = FX keys:", food_keys == fx_keys)
print("Food = Energy keys:", food_keys == energy_keys)
print("Shared key count:", len(food_keys & fx_keys & energy_keys))

food_cols = [
    "geo_id", "price_date", "mkt_name", "adm1_name", "lat", "lon", "currency",
    "wheat", "c_wheat", "c_wheat_flour", "c_rice", "c_sugar", "c_food_price_index",
    "trust_wheat", "trust_wheat_flour", "trust_rice", "trust_sugar",
    "trust_food_price_index", "data_coverage", "data_coverage_recent",
    "index_confidence_score", "spatially_interpolated"
]
fx_cols = [
    "geo_id", "price_date", "exchange_rate_unofficial", "c_exchange_rate_unofficial",
    "inflation_exchange_rate_unofficial", "trust_exchange_rate_unofficial",
    "c_milling_cost_wheat", "c_wage_non_qualified_labour", "c_wage_qualified_labour"
]
energy_cols = [
    "geo_id", "price_date", "fuel_diesel", "c_fuel_diesel", "trust_fuel_diesel",
    "fuel_gas", "c_fuel_gas", "trust_fuel_gas",
    "fuel_petrol_gasoline", "c_fuel_petrol_gasoline", "trust_fuel_petrol_gasoline",
    "spatially_interpolated"
]
food_selected = food[[c for c in food_cols if c in food.columns]].copy()
food_selected = food_selected.rename(columns={
    "spatially_interpolated": "spatially_interpolated_food"
})
fx_selected = fx[[c for c in fx_cols if c in fx.columns]].copy()
energy_selected = energy[[c for c in energy_cols if c in energy.columns]].copy()
energy_selected = energy_selected.rename(columns={
    "spatially_interpolated": "spatially_interpolated_energy"
})

merged = food_selected.merge(
    fx_selected, on=KEY, how="inner", validate="one_to_one"
).merge(
    energy_selected, on=KEY, how="inner", validate="one_to_one"
)
print("Merged shape:", merged.shape)
print("Markets including aggregate:", merged["mkt_name"].nunique())
print("Period:", merged["price_date"].min().date(), "to", merged["price_date"].max().date())
print("Missing key target values:", merged[["c_wheat", "c_exchange_rate_unofficial", "c_fuel_diesel"]].isna().sum().to_dict())


Food = FX keys: True
Food = Energy keys: True
Shared key count: 5175
Merged shape: (5175, 39)
Markets including aggregate: 25
Period: 2009-01-01 to 2026-03-01
Missing key target values: {'c_wheat': 0, 'c_exchange_rate_unofficial': 0, 'c_fuel_diesel': 0}


## 4. Define the forecasting task and build features

At each market-month origin `t`, the model predicts whether `c_wheat` at the **next consecutive month** will be more than 5% higher than the current month. The model is retrained at each forecast origin. Training examples are admitted only when their target month is at or before the current origin date, preventing a future label from crossing a time boundary.

The input price and confidence fields still come from the same retrospective source snapshot; this procedure does not eliminate World Bank vintage revisions.

In [4]:
df = merged[merged["mkt_name"].ne("Market Average")].copy()
df = df.sort_values(["geo_id", "price_date"]).reset_index(drop=True)

price_cols = [
    "c_wheat", "c_wheat_flour", "c_rice", "c_sugar", "c_food_price_index",
    "c_exchange_rate_unofficial", "c_milling_cost_wheat", "c_fuel_diesel",
    "c_fuel_gas", "c_fuel_petrol_gasoline"
]
price_cols = [c for c in price_cols if c in df.columns]
for col in price_cols:
    for lag in (1, 2, 3, 12):
        df[f"{col}_lag{lag}"] = df.groupby("geo_id")[col].shift(lag)

for col in [
    "c_wheat", "c_wheat_flour", "c_rice", "c_exchange_rate_unofficial",
    "c_fuel_diesel", "c_fuel_gas", "c_fuel_petrol_gasoline"
]:
    if col in df.columns:
        for lag in (1, 3, 12):
            df[f"{col}_log_change_{lag}"] = np.log(
                df[col].clip(lower=1e-6) / df[f"{col}_lag{lag}"].clip(lower=1e-6)
            )

quality_cols = [
    "trust_wheat", "trust_wheat_flour", "trust_rice", "trust_sugar",
    "trust_food_price_index", "trust_exchange_rate_unofficial",
    "trust_fuel_diesel", "trust_fuel_gas", "trust_fuel_petrol_gasoline",
    "data_coverage", "data_coverage_recent", "index_confidence_score",
    "spatially_interpolated_food", "spatially_interpolated_energy"
]
quality_cols = [c for c in quality_cols if c in df.columns]
for col in quality_cols:
    df[f"{col}_lag1"] = df.groupby("geo_id")[col].shift(1)

df["month_num"] = df["price_date"].dt.month
df["month_sin"] = np.sin(2 * np.pi * df["month_num"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month_num"] / 12)
df["next_c_wheat"] = df.groupby("geo_id")["c_wheat"].shift(-1)
df["next_date"] = df.groupby("geo_id")["price_date"].shift(-1)
month_gap = (
    (df["next_date"].dt.year - df["price_date"].dt.year) * 12
    + (df["next_date"].dt.month - df["price_date"].dt.month)
)
df = df[month_gap.eq(1)].copy()
df["next_pct_change"] = df["next_c_wheat"] / df["c_wheat"] - 1
df["shock_gt5"] = (df["next_pct_change"] > 0.05).astype(int)
df["previous_month_change"] = df["c_wheat"] / df["c_wheat_lag1"] - 1

core_numeric = list(dict.fromkeys(
    price_cols
    + [f"{c}_lag{lag}" for c in price_cols for lag in (1, 2, 3, 12)
       if f"{c}_lag{lag}" in df.columns]
    + [c for c in df.columns if "_log_change_" in c]
    + ["month_sin", "month_cos"]
))
quality_numeric = quality_cols + [f"{c}_lag1" for c in quality_cols if f"{c}_lag1" in df.columns]
quality_numeric = list(dict.fromkeys(quality_numeric))
enhanced_numeric = core_numeric + [c for c in quality_numeric if c not in core_numeric]
print("Local-market rows for prediction:", len(df))
print("Target positive rate:", round(df["shock_gt5"].mean(), 3))
print("Core numeric features:", len(core_numeric), "| Enhanced numeric features:", len(enhanced_numeric))


Local-market rows for prediction: 4944
Target positive rate: 0.183
Core numeric features: 73 | Enhanced numeric features: 101


## 5. Monthly walk-forward replay

For each month, train only on prior examples whose next-month outcome is already known. The threshold is selected on the 2019–2021 validation period by maximum F1, then fixed for the 2022–March 2026 replay.

In [5]:
# The monthly retraining function is provided for a full rerun.
# Full model fitting can take several minutes. By default, the package reuses the exact
# saved per-origin prediction files so the tables can be reproduced immediately.
# Set RETRAIN_MODELS = True to rebuild both prediction series from source data.

RETRAIN_MODELS = False

def run_prequential_lightgbm(numeric_features, label):
    all_features = numeric_features + ["geo_id"]
    predictions = []
    origins = sorted(
        d for d in df["price_date"].dropna().unique()
        if pd.Timestamp("2018-12-01") <= pd.Timestamp(d) <= pd.Timestamp("2026-02-01")
    )
    for i, origin in enumerate(origins):
        origin = pd.Timestamp(origin)
        # No training example enters until its next-month outcome is known by origin.
        train = df[df["next_date"] <= origin].copy()
        current = df[df["price_date"].eq(origin)].copy()
        if len(train) < 500 or current.empty:
            continue
        x_train = train[all_features].copy()
        x_current = current[all_features].copy()
        x_train["geo_id"] = x_train["geo_id"].astype(str).astype("category")
        categories = x_train["geo_id"].cat.categories
        x_current["geo_id"] = pd.Categorical(
            x_current["geo_id"].astype(str), categories=categories
        )
        model = LGBMClassifier(
            n_estimators=60, num_leaves=12, learning_rate=0.04,
            min_child_samples=35, subsample=0.9, colsample_bytree=0.8,
            reg_lambda=4, class_weight="balanced", verbosity=-1,
            n_jobs=2, random_state=42,
        )
        model.fit(x_train, train["shock_gt5"], categorical_feature=["geo_id"])
        current[f"prob_{label}"] = model.predict_proba(x_current)[:, 1]
        predictions.append(current[[
            "geo_id", "mkt_name", "price_date", "next_date",
            "shock_gt5", "next_pct_change", "previous_month_change", f"prob_{label}"
        ]])
        if i % 12 == 0:
            print(f"{label}: origin {origin:%Y-%m}; train rows={len(train)}; current rows={len(current)}")
    if not predictions:
        raise RuntimeError(f"No prequential predictions generated for {label}.")
    return pd.concat(predictions, ignore_index=True)

core_cache = Path("marketshield_prequential_core_predictions.csv")
enhanced_cache = Path("marketshield_prequential_enhanced_predictions.csv")
if not RETRAIN_MODELS and core_cache.exists() and enhanced_cache.exists():
    print("Loading saved monthly prequential predictions. Set RETRAIN_MODELS=True to refit models.")
    preq_core = pd.read_csv(core_cache, parse_dates=["price_date", "next_date"])
    preq_enhanced = pd.read_csv(enhanced_cache, parse_dates=["price_date", "next_date"])
else:
    print("Running core-feature LightGBM for each forecast origin...")
    preq_core = run_prequential_lightgbm(core_numeric, "core")
    print("Running LightGBM with additional data-quality features...")
    preq_enhanced = run_prequential_lightgbm(enhanced_numeric, "enhanced")

join_keys = ["geo_id", "price_date", "next_date"]
preq = preq_enhanced.merge(
    preq_core[join_keys + ["prob_core"]], on=join_keys, how="inner", validate="one_to_one",
    suffixes=("", "_core")
)
if "shock_gt5_core" in preq and not (preq["shock_gt5"] == preq["shock_gt5_core"]).all():
    raise ValueError("Cached/model prediction labels do not match across feature variants.")
if "shock_gt5_core" in preq:
    preq = preq.drop(columns=["shock_gt5_core"])
print("Prequential predictions:", preq.shape)
display(preq.head())

Loading saved monthly prequential predictions. Set RETRAIN_MODELS=True to refit models.
Prequential predictions: (2088, 9)


,geo_id,price_date,next_date,mkt_name,shock_gt5,next_pct_change,previous_month_change,prob_enhanced,prob_core
0,gid_126500000540300000,2018-12-01,2019-01-01,Soqatra (Hudaibo),0,0.000000,0.034483,0.639693,0.629303
1,gid_127900000450200000,2018-12-01,2019-01-01,Aden City,0,-0.033816,-0.050459,0.404044,0.412921
2,gid_130600000448800000,2018-12-01,2019-01-01,Al Hawtah- Main road - Back Street,1,0.063231,0.083333,0.404887,0.408827
3,gid_131300000453800000,2018-12-01,2019-01-01,Zungubar City,0,0.022409,-0.088235,0.360317,0.343543
4,gid_135800000440200000,2018-12-01,2019-01-01,Taiz City,1,0.053690,-0.018706,0.376364,0.425866


## 6. Choose the threshold on validation only; report the later period separately

Validation outcomes are target months from January 2019 through December 2021. The retrospective test outcomes are target months from January 2022 through March 2026. The later period is not a completely untouched holdout because prior experiments already inspected aggregated results for overlapping dates.

In [6]:
validation = preq[
    (preq["next_date"] >= pd.Timestamp("2019-01-01"))
    & (preq["next_date"] < pd.Timestamp("2022-01-01"))
].copy()
test = preq[preq["next_date"] >= pd.Timestamp("2022-01-01")].copy()

if validation.empty or test.empty:
    raise ValueError("Validation or test split is empty; inspect the source date coverage.")

threshold_rows = []
selected_thresholds = {}
for model_label, prob_col in [("core", "prob_core"), ("enhanced", "prob_enhanced")]:
    for threshold in np.arange(0.10, 0.801, 0.025):
        pred = (validation[prob_col].to_numpy() >= threshold).astype(int)
        y = validation["shock_gt5"].to_numpy()
        threshold_rows.append({
            "model": model_label,
            "threshold": float(threshold),
            "precision": precision_score(y, pred, zero_division=0),
            "recall": recall_score(y, pred, zero_division=0),
            "f1": f1_score(y, pred, zero_division=0),
            "alerts": int(pred.sum()),
        })

threshold_table = pd.DataFrame(threshold_rows)
for model_label in ("core", "enhanced"):
    subset = threshold_table[threshold_table["model"].eq(model_label)]
    best = subset.sort_values(["f1", "precision"], ascending=False).iloc[0]
    selected_thresholds[model_label] = float(best["threshold"])

print("Validation rows:", len(validation), "| Test rows:", len(test))
print("Positive rate — validation:", round(validation["shock_gt5"].mean(), 3))
print("Positive rate — test:", round(test["shock_gt5"].mean(), 3))
print("Thresholds chosen from validation:", selected_thresholds)
display(
    threshold_table[
        threshold_table["model"].eq("enhanced")
        & threshold_table["threshold"].round(3).isin([0.4, 0.45, 0.5, 0.55, 0.6])
    ].round(3)
)

def classification_report_row(name, frame, prob_col, threshold):
    y = frame["shock_gt5"].to_numpy()
    prob = frame[prob_col].to_numpy()
    pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "Model / split": name,
        "ROC-AUC": roc_auc_score(y, prob),
        "Average precision": average_precision_score(y, prob),
        "Threshold (validation-selected)": threshold,
        "Precision": precision_score(y, pred, zero_division=0),
        "Recall": recall_score(y, pred, zero_division=0),
        "F1": f1_score(y, pred, zero_division=0),
        "Balanced accuracy": balanced_accuracy_score(y, pred),
        "Alerts": int(pred.sum()), "TP": int(tp), "FP": int(fp),
        "FN": int(fn), "TN": int(tn),
    }

summary_rows = []
for model_label, prob_col, pretty in [
    ("core", "prob_core", "LightGBM — core features"),
    ("enhanced", "prob_enhanced", "LightGBM — plus data-quality features"),
]:
    threshold = selected_thresholds[model_label]
    summary_rows.append(classification_report_row(pretty + " — validation", validation, prob_col, threshold))
    summary_rows.append(classification_report_row(pretty + " — later-period replay", test, prob_col, threshold))
summary = pd.DataFrame(summary_rows)
display(summary.round(3))

# Interpretable momentum rule: flag when the current month's c_wheat rose >5% from its prior month.
rule_summary = []
for split_name, frame in [("Momentum rule — validation", validation), ("Momentum rule — later-period replay", test)]:
    y = frame["shock_gt5"].to_numpy()
    pred = (frame["previous_month_change"].to_numpy() > 0.05).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    rule_summary.append({
        "Model / rule": split_name,
        "Precision": precision_score(y, pred, zero_division=0),
        "Recall": recall_score(y, pred, zero_division=0),
        "F1": f1_score(y, pred, zero_division=0),
        "Alerts": int(pred.sum()), "TP": int(tp), "FP": int(fp),
        "FN": int(fn), "TN": int(tn),
    })
display(pd.DataFrame(rule_summary).round(3))

preq.to_csv("marketshield_prequential_predictions.csv", index=False)
summary.to_csv("marketshield_prequential_summary.csv", index=False)
print("Saved marketshield_prequential_predictions.csv and marketshield_prequential_summary.csv")


Validation rows: 864 | Test rows: 1224
Positive rate — validation: 0.214
Positive rate — test: 0.127
Thresholds chosen from validation: {'core': 0.47499999999999987, 'enhanced': 0.4999999999999999}


,model,threshold,precision,recall,f1,alerts
41,enhanced,0.40,0.337,0.735,0.462,404
43,enhanced,0.45,0.390,0.670,0.493,318
45,enhanced,0.50,0.459,0.611,0.524,246
47,enhanced,0.55,0.489,0.503,0.496,190
49,enhanced,0.60,0.500,0.405,0.448,150


,Model / split,ROC-AUC,Average precision,Threshold (validation-selected),Precision,Recall,F1,Balanced accuracy,Alerts,TP,FP,FN,TN
0,LightGBM — core features — validation,0.749,0.474,0.475,0.435,0.670,0.528,0.717,285,124,161,61,518
1,LightGBM — core features — later-period replay,0.684,0.219,0.475,0.224,0.555,0.319,0.638,384,86,298,69,771
2,LightGBM — plus data-quality features — valida...,0.747,0.480,0.500,0.459,0.611,0.524,0.707,246,113,133,72,546
3,LightGBM — plus data-quality features — later-...,0.680,0.218,0.500,0.223,0.510,0.310,0.626,355,79,276,76,793


,Model / rule,Precision,Recall,F1,Alerts,TP,FP,FN,TN
0,Momentum rule — validation,0.462,0.427,0.444,171,79,92,106,587
1,Momentum rule — later-period replay,0.212,0.226,0.219,165,35,130,120,939


Saved marketshield_prequential_predictions.csv and marketshield_prequential_summary.csv


## 7. Interpretation and operational-readiness gate

### Current result: research prototype only

Two LightGBM variants were compared with the same monthly prequential protocol, target and time windows. The **core** model uses price lags, log changes and seasonality; the **expanded** model adds lagged source confidence/data-coverage indicators. Thresholds are selected by F1 on the 2019–2021 validation predictions and then frozen for the 2022–March 2026 replay.

The core model selected a validation threshold of about 0.475. Its validation ROC-AUC/AP/precision/recall/F1 were approximately 0.749/0.474/0.435/0.670/0.528; on the later-period replay they were 0.684/0.219/0.224/0.555/0.319, with 384 alerts (86 true positives, 298 false positives, 69 false negatives, 771 true negatives).

The expanded model selected a threshold of about 0.50. Its validation ROC-AUC/AP/precision/recall/F1 were approximately 0.747/0.480/0.459/0.611/0.524; on the later replay they were 0.680/0.218/0.223/0.510/0.310, with 355 alerts (79 true positives, 276 false positives, 76 false negatives, 793 true negatives). The core model was marginally better by validation F1 and replay F1; adding quality indicators did not yield an overall gain. Both models produce many false alerts (about 78% of alerts on replay).

A simple rule that flags a >5% rise in the current month had replay F1 around 0.219 (165 alerts, 35 true positives and 130 false positives). LightGBM increased recall, but only by issuing substantially more alerts; this is not enough evidence for operational value.

### Why the later period is not fully independent

The 2022–March 2026 dates were inspected in earlier project experiments. This monthly replay limits target-time leakage, but it does not undo earlier inspection or the potential revision of historical source estimates. These results are a **retrospective walk-forward replay**, not an untouched independent test. The stored prediction CSVs let the reported summary be reviewed without rerunning every monthly fit; set `RETRAIN_MODELS = True` in the model cell to refit both variants.

### What is required before any live use

1. Obtain point-in-time snapshots or independent market observations with collection dates; this source release ends in March 2026, and historical estimates can be revised.
2. Freeze models and thresholds before evaluating on a future, uninspected period or independent observed prices.
3. Predefine the alert objective and the costs of false alarms and missed events.
4. Report performance by market and period, confidence intervals, probability calibration, and data-quality failure cases.
5. Run in shadow mode with human review and no public/humanitarian alerts before considering any pilot.

**Conclusion:** keep Yemen MarketShield AI at research-prototype status. The label is a >5% rise in the next month’s `c_wheat` close estimate; it is not an official food crisis or humanitarian outcome. Do not use predictions for operational decisions.

### External source discovery

The Yemen Ministry of Industry, Trade and Investment publishes weekly price-list observations for wheat grain and wheat flour (for example, the release dated 17 September 2026), but the retrieved table does not match this model's per-market monthly `c_wheat` target. The WFP food-price database documented by HDX HAPI has monthly price series, market identifiers, price type, unit, currency, and reference period fields. It is the next candidate to inspect for an independent holdout, but its coverage after March 2026 and overlap with the World Bank estimates' upstream data must be confirmed before calling it independent. API use requires an application identifier. See https://moit.gov.ye/complaints/prices and https://hdx-hapi.readthedocs.io/en/latest/data_usage_guides/food_security_nutrition_and_poverty/.